# MoveTech — Otimização de Frota e Tempo de Espera

## Projeto Integrador de Ciência de Dados

**Componente:** Matemática e Estatística para Ciência de Dados  
**Integrantes:** Danilo Ribeiro Rodrigues Junior; Giovanna Muniz Antoniazzi; Leonardo Ricardo Belido; Nicollas Bernardo Furtunatto; Yasmin Alves Vitório

## Objetivo

Analisar os dados de solicitações de corridas da MoveTech para apoiar decisões sobre a distribuição da frota, o tempo de espera dos passageiros e a aceitação das corridas.

O trabalho utiliza preparação de dados, amostragem, estatística descritiva, probabilidade e inferência estatística.

## Perguntas orientadoras

- Como as categorias de serviço se relacionam com a aceitação das corridas?
- Qual é o tempo médio de espera e a probabilidade de ultrapassar 10 minutos?
- Quais regiões e horários exigem maior atenção na distribuição da frota?
- Existem corridas com comportamento anômalo?

## 1. Preparação do ambiente

O projeto utiliza pandas para manipular tabelas, NumPy para apoiar o sorteio e openpyxl para permitir a leitura de arquivos Excel pelo pandas.

A célula de instalação abaixo precisa ser executada somente quando o openpyxl não estiver instalado no ambiente do kernel selecionado.

In [ ]:
%pip install openpyxl

## 2. Origem e preparação dos dados

A base original `DADOS/movetech_corridas.csv` contém 19.202 registros e 13 colunas. Cada linha representa um registro de solicitação de corrida.

A preparação inicial foi realizada no Power Query do Excel, preservando o CSV original. Foram realizadas as seguintes operações:

- Remoção de 91 linhas completamente duplicadas, considerando as 13 colunas.
- Padronização de espaços, capitalização, acentuação e abreviações nas variáveis categóricas.
- Padronização dos separadores decimais e remoção do símbolo monetário.
- Conversão de tempo de espera, duração, distância e valor para números decimais.
- Padronização e conversão dos horários, incluindo formatos como `1h02`.

A tabela resultante foi salva em `DADOS/DADOS.xlsx`, com 19.111 registros e 13 colunas.

As etapas estão registradas na consulta do Power Query. Para atualizá-la em outro computador, pode ser necessário ajustar o caminho da etapa Fonte para o CSV original.

Este notebook lê a tabela já salva no Excel; a leitura não executa nem atualiza a consulta do Power Query.

In [ ]:
import pandas as pd
from pathlib import Path

# Caminho da planilha tratada no Power Query.
caminho_dados = Path("DADOS") / "DADOS.xlsx"

# Lê a primeira aba da planilha.
dados = pd.read_excel(caminho_dados, sheet_name=0)

# Confere o tamanho da tabela.
print(f"Quantidade de registros: {dados.shape[0]}")
print(f"Quantidade de colunas: {dados.shape[1]}")

# Mostra as primeiras cinco linhas.
dados.head()

## 3. Conferência da importação

Verificamos o número de registros, as colunas, os tipos reconhecidos pelo pandas e a quantidade de valores preenchidos.

O tipo computacional indica como o dado está armazenado. Ele não determina, sozinho, a classificação estatística da variável.

Também inspecionamos um valor de `hora_solicitacao` para verificar como o horário foi importado.

In [ ]:
# Mostra os tipos das colunas e a quantidade de valores preenchidos.
dados.info()

In [ ]:
# Mostra o primeiro horário e o tipo desse valor.
horario = dados["hora_solicitacao"].iloc[0]

print("Primeiro horário:", horario)
print("Tipo do valor:", type(horario))

### Resultado da conferência

A planilha foi importada com 19.111 registros e 13 colunas. Todas as colunas apresentaram 19.111 valores não nulos.

As colunas `tempo_espera`, `duracao_viagem`, `distancia_km` e `valor` foram reconhecidas como números decimais.

O primeiro valor de `hora_solicitacao` foi reconhecido como `datetime.time`. Assim, a indicação `object` nessa coluna não representa, por si só, um erro.

A ausência de valores nulos não garante a ausência de textos vazios, inconsistências de negócio ou valores atípicos. Essas verificações devem ser consideradas separadamente.

## 4. Classificação das variáveis

| Variável | Classificação | Justificativa |
|---|---|---|
| id_corrida | Identificador nominal | Identifica o registro, sem representar uma medida. |
| bairro_origem | Qualitativa nominal | Identifica bairros, sem hierarquia. |
| regiao | Qualitativa nominal | Identifica regiões, sem escala de avaliação. |
| hora_solicitacao | Temporal cíclica | Representa o horário dentro do ciclo diário. |
| dia_semana | Qualitativa com organização cíclica | Identifica dias que se repetem semanalmente. |
| categoria_servico | Qualitativa nominal | Identifica modalidades sem uma hierarquia única entre todas elas. |
| cor_veiculo | Qualitativa nominal | Identifica cores, sem ordenação. |
| nota_motorista | Qualitativa ordinal | Representa níveis ordenados de avaliação. |
| tempo_espera | Quantitativa contínua | Mede o tempo de espera em minutos. |
| duracao_viagem | Quantitativa contínua | Mede a duração da viagem em minutos. |
| distancia_km | Quantitativa contínua | Mede a distância percorrida. |
| valor | Quantitativa, tratada como contínua na análise | Representa um valor monetário registrado em centavos. |
| status | Qualitativa nominal | Identifica desfechos diferentes, sem hierarquia. |

### Preservação da ordem das avaliações

A nota do motorista será representada como uma categoria ordenada.

Adotamos provisoriamente a escala de 1 a 5, a confirmar com a documentação da base ou com o professor. Os valores observados são 2, 3, 4 e 5.

Incluir a categoria 1 não cria registros novos: apenas registra um nível possível da escala adotada.

Uma nota maior representa uma avaliação superior, mas não permite afirmar que nota 4 corresponde ao dobro da qualidade de nota 2.

In [ ]:
# Define as categorias e a ordem das avaliações.
tipo_nota = pd.CategoricalDtype(
    categories=[1, 2, 3, 4, 5],
    ordered=True
)

# Converte a coluna para categoria ordenada.
dados["nota_motorista"] = dados["nota_motorista"].astype(tipo_nota)

# Confere o tipo e as quantidades.
print("Tipo da coluna:", dados["nota_motorista"].dtype)
print(dados["nota_motorista"].value_counts().sort_index())

## 5. Distribuição temporal e definição dos grupos

Extraímos a hora de cada solicitação para contar os registros nos intervalos de uma hora.

A contagem inclui todos os status, pois representa solicitações, e não apenas corridas aceitas.

Essa descrição da base completa será usada para planejar a amostragem. Um volume elevado de solicitações não demonstra, sozinho, maior tempo de espera.

In [ ]:
# Extrai a hora de cada solicitação.
dados["hora"] = dados["hora_solicitacao"].apply(
    lambda horario: horario.hour
)

# Conta as solicitações por hora e inclui horas sem registros.
solicitacoes_por_hora = (
    dados["hora"]
    .value_counts()
    .reindex(range(24), fill_value=0)
)

print(solicitacoes_por_hora)

### Agrupamento por região e período

Dividimos o dia em quatro intervalos de seis horas:

- Madrugada: 00h até antes das 06h.
- Manhã: 06h até antes das 12h.
- Tarde: 12h até antes das 18h.
- Noite: 18h até antes das 24h.

Essa divisão é uma convenção de organização, não uma definição de horário de pico.

O cruzamento das três regiões com os quatro períodos produz 12 grupos. Eles serão utilizados como estratos porque o problema de distribuição da frota envolve tanto a localização quanto o momento da solicitação.

In [ ]:
# Agrupa a hora da solicitação em quatro períodos.
dados["faixa_horaria"] = pd.cut(
    dados["hora"],
    bins=[0, 6, 12, 18, 24],
    labels=["Madrugada", "Manhã", "Tarde", "Noite"],
    right=False
)

# Conta as solicitações por região e período.
distribuicao = pd.crosstab(
    dados["regiao"],
    dados["faixa_horaria"],
    margins=True,
    margins_name="Total"
)

display(distribuicao)

### Distribuição das categorias de serviço

Contamos os registros e calculamos a participação de cada serviço na base completa.

Essa verificação permite identificar categorias menos frequentes, que podem ter poucas observações após a amostragem.

A categoria de serviço não será utilizada para definir os estratos desta proposta. Por isso, sua representação será conferida depois dos sorteios.

In [ ]:
# Mostra a distribuição dos serviços na base completa.
distribuicao_servicos = (
    dados["categoria_servico"]
    .value_counts()
    .rename_axis("categoria_servico")
    .reset_index(name="quantidade")
)

distribuicao_servicos["percentual"] = (
    distribuicao_servicos["quantidade"] / len(dados) * 100
)

display(distribuicao_servicos.round(2))

## 6. Plano de amostragem estratificada

Escolhemos uma amostra de 2.000 registros, acima do mínimo de 300 exigido na atividade.

A escolha busca incluir mais observações nos grupos menores e reduzir a incerteza em comparação com amostras menores. Esse tamanho não foi calculado para garantir uma margem de erro específica.

Os estratos são as combinações de região e faixa horária. A alocação será proporcional: cada grupo receberá uma quantidade de registros próxima da sua participação na base.

Para cada estrato:

Quantidade ideal na amostra = (quantidade do estrato na base / total da base) × 2.000.

In [ ]:
# Tamanho escolhido para a amostra.
tamanho_amostra = 2000

# Conta os registros de cada combinação de região e período.
plano_amostral = (
    dados.groupby(["regiao", "faixa_horaria"], observed=True)
    .size()
    .reset_index(name="quantidade_base")
)

# Calcula a participação de cada grupo na base.
plano_amostral["proporcao"] = (
    plano_amostral["quantidade_base"] / len(dados)
)

# Calcula a quantidade proporcional ideal para a amostra.
plano_amostral["quantidade_ideal"] = (
    plano_amostral["proporcao"] * tamanho_amostra
)

display(plano_amostral)

### Ajuste para quantidades inteiras

As quantidades proporcionais ideais podem conter casas decimais, mas não é possível selecionar uma fração de registro.

Aplicamos o método dos maiores restos:

1. Atribuir a cada grupo a parte inteira da quantidade ideal.
2. Calcular quantos registros faltam para completar 2.000.
3. Distribuir os registros restantes aos grupos com as maiores partes decimais.

Esse procedimento mantém o total exato de 2.000 e aproxima a distribuição planejada das proporções da base.

In [ ]:
# Guarda a parte inteira da quantidade ideal.
plano_amostral["quantidade_amostra"] = (
    plano_amostral["quantidade_ideal"].astype(int)
)

# Calcula a parte decimal que sobrou.
plano_amostral["resto"] = (
    plano_amostral["quantidade_ideal"]
    - plano_amostral["quantidade_amostra"]
)

# Descobre quantos registros faltam para completar 600.
faltantes = tamanho_amostra - plano_amostral["quantidade_amostra"].sum()

# Identifica os grupos com os maiores restos.
indices_prioritarios = (
    plano_amostral["resto"].nlargest(int(faltantes)).index
)

# Acrescenta um registro a cada grupo prioritário.
plano_amostral.loc[indices_prioritarios, "quantidade_amostra"] += 1

display(
    plano_amostral[
        ["regiao", "faixa_horaria", "quantidade_base", "quantidade_amostra"]
    ]
)

print("Total planejado:", plano_amostral["quantidade_amostra"].sum())

### Sorteio dentro dos estratos

Dentro de cada um dos 12 grupos, selecionamos aleatoriamente a quantidade definida no plano.

O sorteio é realizado sem reposição: a mesma linha não pode ser selecionada duas vezes.

Utilizamos a semente 42 para permitir a reprodução do sorteio com os mesmos dados, na mesma ordem e com o mesmo ambiente de execução.

Ao final, reunimos os grupos e verificamos o tamanho da amostra e a existência de identificadores repetidos.

In [ ]:
import numpy as np

# Controla o sorteio para permitir sua reprodução.
gerador = np.random.default_rng(42)

# Guarda as partes sorteadas de cada grupo.
partes_amostra = []

for grupo in plano_amostral.itertuples(index=False):
    # Seleciona as corridas da região e do período deste grupo.
    filtro = (
        (dados["regiao"] == grupo.regiao)
        & (dados["faixa_horaria"] == grupo.faixa_horaria)
    )

    dados_grupo = dados.loc[filtro]

    # Sorteia a quantidade planejada, sem repetir registros.
    parte = dados_grupo.sample(
        n=int(grupo.quantidade_amostra),
        replace=False,
        random_state=gerador
    )

    partes_amostra.append(parte)

# Reúne os 12 grupos em uma única tabela.
amostra_estratificada = pd.concat(
    partes_amostra,
    ignore_index=True
)

print("Registros sorteados:", len(amostra_estratificada))
print(
    "IDs repetidos:",
    amostra_estratificada["id_corrida"].duplicated().sum()
)

## 7. Amostragem aleatória simples

Selecionamos 2.000 registros diretamente da base completa, sem reposição e com semente 42.

Nesse método, cada registro tem a mesma probabilidade de seleção. Entretanto, não há uma quantidade reservada para cada região ou período.

Mantemos o mesmo tamanho das duas amostras para comparar os métodos em condições equivalentes.

As duas amostras podem compartilhar registros, pois são seleções realizadas sobre a mesma base.

In [ ]:
# Sorteia diretamente da base completa.
amostra_aleatoria = dados.sample(
    n=tamanho_amostra,
    replace=False,
    random_state=42
).reset_index(drop=True)

print("Registros sorteados:", len(amostra_aleatoria))
print(
    "IDs repetidos:",
    amostra_aleatoria["id_corrida"].duplicated().sum()
)

## 8. Comparação da representatividade

Comparamos a participação percentual dos 12 grupos na base completa e nas duas amostras.

Para cada grupo, calculamos o desvio absoluto:

Desvio = |percentual na amostra − percentual na base|.

O resultado é expresso em pontos percentuais. Quanto menor o desvio, mais próxima está a participação do grupo na amostra em relação à base.

Essa comparação avalia a composição por região e período; não garante representatividade em todas as características.

In [ ]:
# Colunas que definem nossos grupos.
grupos = ["regiao", "faixa_horaria"]

# Calcula a participação percentual de cada grupo.
percentual_base = (
    dados.groupby(grupos, observed=True).size()
    / len(dados) * 100
)

percentual_estratificada = (
    amostra_estratificada.groupby(grupos, observed=True).size()
    / len(amostra_estratificada) * 100
)

percentual_aleatoria = (
    amostra_aleatoria.groupby(grupos, observed=True).size()
    / len(amostra_aleatoria) * 100
)

# Reúne os resultados lado a lado.
comparacao = pd.DataFrame({
    "base_%": percentual_base,
    "estratificada_%": percentual_estratificada,
    "aleatoria_%": percentual_aleatoria
}).fillna(0)

# Calcula a distância entre cada amostra e a base.
comparacao["desvio_estratificada_pp"] = (
    comparacao["estratificada_%"] - comparacao["base_%"]
).abs()

comparacao["desvio_aleatoria_pp"] = (
    comparacao["aleatoria_%"] - comparacao["base_%"]
).abs()

display(comparacao.round(3))

### Verificação das categorias de serviço

Comparamos as quantidades e os percentuais dos serviços na base e nas duas amostras.

Essa verificação complementa a análise dos estratos, pois os serviços não foram usados para controlar o sorteio.

Além das proporções, observamos as contagens: categorias com poucos registros podem apresentar estimativas menos precisas, especialmente quando subdivididas por região, horário ou status.

In [ ]:
# Conta os registros de cada serviço na base e nas amostras.
comparacao_servicos = pd.DataFrame({
    "base_n": dados["categoria_servico"].value_counts(),
    "estratificada_n": (
        amostra_estratificada["categoria_servico"].value_counts()
    ),
    "aleatoria_n": (
        amostra_aleatoria["categoria_servico"].value_counts()
    )
}).fillna(0).astype(int)

# Calcula os percentuais em cada conjunto.
comparacao_servicos["base_%"] = (
    comparacao_servicos["base_n"] / len(dados) * 100
)

comparacao_servicos["estratificada_%"] = (
    comparacao_servicos["estratificada_n"]
    / len(amostra_estratificada) * 100
)

comparacao_servicos["aleatoria_%"] = (
    comparacao_servicos["aleatoria_n"]
    / len(amostra_aleatoria) * 100
)

display(comparacao_servicos.round(2))

## 9. Escolha da amostra e limitações

Adotaremos a amostra estratificada para as próximas análises, pois preservou melhor as proporções dos grupos de região e período, diretamente relacionados ao problema de distribuição da frota.

Neste sorteio, os desvios desses grupos ficaram abaixo de 0,03 ponto percentual na estratificada. Na aleatória simples, o maior desvio foi aproximadamente 1,035 ponto percentual.

Essa vantagem é esperada, pois a amostra estratificada foi construída para controlar essas proporções. O resultado não demonstra superioridade em todas as variáveis ou em todos os possíveis sorteios.

Todas as categorias de serviço apareceram nas duas amostras. O MoveTech Bag, menos frequente na base, apresentou 44 registros na estratificada. Sua presença não garante precisão suficiente para análises detalhadas desse serviço.

Outras limitações:

- A amostragem não corrige possíveis problemas da base original.
- Os registros representam solicitações, não motoristas ou passageiros distintos.
- As proporções por estrato são aproximadas devido ao arredondamento das quantidades.
- As inferências posteriores deverão considerar o desenho amostral.
- As quatro faixas horárias ainda não definem os horários de pico.